# Track A — S24 SD LoRA VS Generation (Colab)

**전제조건**: 런타임 유형 → GPU (T4 이상)

**순서**: 셀을 위에서 아래로 순서대로 실행

In [ ]:
# [1] GPU 확인
import torch
print(f'torch {torch.__version__}')
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경 > GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# [2] 패키지 설치
# Colab 기본 torch는 cu130 빌드지만 torchvision은 cu128 → 버전 충돌
# 공식 PyTorch cu128 빌드로 통일 (cu128은 cu130 하드웨어에서 하위 호환)
import subprocess, importlib.util, os

WHL = 'https://download.pytorch.org/whl/cu128'

# Step 1: torch + torchvision 공식 cu128 빌드로 교체
subprocess.run(['pip', 'install', '-q',
                'torch', 'torchvision',
                '--index-url', WHL], check=True)

# Step 2: peft / diffusers 설치
subprocess.run(['pip', 'install', '-q',
                'peft', 'diffusers', 'accelerate', 'transformers', 'h5py'], check=True)

# Step 3: peft 패치 — old torchao에서 raise 대신 return False
_spec = importlib.util.find_spec('peft')
_utils = os.path.join(os.path.dirname(_spec.origin), 'import_utils.py')
with open(_utils) as f:
    _src = f.read()
_patched = _src.replace(
    'if torchao_version < TORCHAO_MINIMUM_VERSION:',
    'if torchao_version < TORCHAO_MINIMUM_VERSION:\n        return False  # patched'
)
if _patched != _src:
    with open(_utils, 'w') as f: f.write(_patched)
    print('peft patched OK')
else:
    print('peft patch: 패턴 없음 (이미 적용됐거나 버전 다름)')

# Step 4: 동작 확인
import torch, torchvision, peft, diffusers
print(f'torch {torch.__version__}')
print(f'torchvision {torchvision.__version__} OK')
print(f'peft {peft.__version__}, diffusers {diffusers.__version__}')

In [ ]:
# [3] 코드 clone (S24 .npz 데이터 포함)
import os, subprocess

REPO    = 'https://github.com/heegyukim4043/test_diffusion_EEG_visualstim_image.git'
WORKDIR = '/content/vsvi_project'

if not os.path.exists(WORKDIR):
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)
else:
    subprocess.run(['git', '-C', WORKDIR, 'pull', 'origin', 'main'], check=True)

os.chdir(WORKDIR)
subprocess.run(['git', 'log', '--oneline', '-3'])

npz = sorted([f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')])
print(f'S24 .npz: {len(npz)}개  SupCon ckpt: {os.path.exists("checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt")}')

In [ ]:
# [4] Google Drive 마운트 (체크포인트 저장용)
from google.colab import drive
drive.mount('/content/drive')

CKPT_DRIVE = '/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen'
os.makedirs(CKPT_DRIVE, exist_ok=True)

dst = f'{WORKDIR}/checkpoints_vsre_lora_gen'
if not os.path.exists(dst):
    os.symlink(CKPT_DRIVE, dst)
print(f'checkpoints_vsre_lora_gen -> {CKPT_DRIVE}')

In [ ]:
# [5] 환경 확인
import torch, os
checks = {
    'CUDA': torch.cuda.is_available(),
    'npz 10개': len([f for f in os.listdir('preproc_vs_re') if 'subj_24' in f and f.endswith('.npz')]) == 10,
    'SupCon ckpt': os.path.exists('checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon/subj24_best.pt'),
    'images': os.path.exists('preproc_data_vi/images/01.png'),
    'ckpt_root': os.path.islink('checkpoints_vsre_lora_gen') or os.path.isdir('checkpoints_vsre_lora_gen'),
}
for k, v in checks.items():
    print(f'  [{"OK" if v else "FAIL"}] {k}')
assert all(checks.values()), 'FAIL 항목 확인 필요'

In [ ]:
# [6] S24 LoRA r=16 학습 (T4 기준 약 8~12시간)
import subprocess, os
os.chdir('/content/vsvi_project')
result = subprocess.run([
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '16',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
])
print(f'Exit code: {result.returncode}')

In [ ]:
# [7] S24 LoRA r=32 학습 (r=16 완료 후 실행)
import subprocess, os
os.chdir('/content/vsvi_project')
result = subprocess.run([
    'python', 'train_vs_re_lora_gen.py',
    '--subject_ids', '24',
    '--lora_r', '32',
    '--n_eeg_tokens', '16',
    '--epochs', '100',
    '--img_root', 'preproc_data_vi/images',
    '--supcon_ckpt', 'checkpoints_vsre_dino/20260604_091352_ch32_merged_ep200_supcon',
    '--ckpt_root', 'checkpoints_vsre_lora_gen',
])
print(f'Exit code: {result.returncode}')

In [ ]:
# [8] 결과 확인
import glob
results = glob.glob('/content/drive/MyDrive/vsvi_data/checkpoints_vsre_lora_gen/**/results_dual_gallery.csv', recursive=True)
for r in sorted(results):
    print(r)
    with open(r) as f: print(f.read())